<strong># OVERVIEW OF THE STUDY</strong>
<br><br>
<strong>## Parameters of the absorber-location sweep</strong>
<br><br>
<strong>Material A refractive index</strong><br>
n<sub>A</sub> ∈ [1.01, 1.9] (20 steps)
<br><br>
<strong>Material B refractive index</strong><br>
n<sub>B</sub> = 1.0
<br><br>
<strong>Absorber extinction coefficient</strong><br>
k ∈ {0.01, 0.10} (10 steps)
<br><br>
<strong>Absorber location</strong><br>
(i) Core only<br>
(ii) Low-index layers only (material B)<br>
(iii) All Bragg layers (materials A and B)<br>
(iv) Additional outer absorbing shell
<br><br>
<strong>Design wavelength</strong><br>
λ<sub>design</sub> ∈ {480, 560, 640} nm
<br><br>
<strong>Number of Bragg layers</strong><br>
N = 60
<br><br>
<strong>Outer Bragg layer</strong><br>
Material A
<br><br>
<strong>Collection geometry</strong><br>
Backward hemisphere collection
<br><br>
<strong>Collection numerical aperture</strong><br>
NA = 1.0
<br><br>
<strong>Outer shell thicknesses</strong><br>
d<sub>shell</sub> ∈ {5, 10, 20, 40} nm
<br><br>
<strong>Core thickness Factors</strong><br>
d<sub>shell</sub> ∈ {0.5, 1.0, 2.0, 4.0}
<br><br>
<strong>Objective</strong><br>
Quantify how absorber placement influences the optical response, colour quality, scattering efficiency, and collection efficiency of Bragg onion photonic pigments while keeping the absorber strength, geometry, and collection conditions fixed.


## Quantities to analyze


### 1. Colour performance

Rosch–MacAdam colour solid

Optimal colour contour

CIELAB a*–b*

Colour strips

η_C, η_L heatmaps

### 2. Scattering

Differential scattering vs θ (dB)

Integrated efficiencies vs λ

Asymmetry g, albedo

Forward / side / backward (dB)

Pie chart at α = 70.5°

### 3. Near fields

|E| (dB), xz / yz / xy

Poynting flow + streamlines

Hotspots

### 4. Resonances

Mie l‑decomposition

l‑resolved spectra

### 5. Sensitivity

η_C, η_L vs n_layers, Δn, k

In [ ]:
# CELL 0 — IMPORTS & GLOBAL SETTINGS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from bragg_onion._notebook_tools.io import load_sweep_outputs
from bragg_onion._notebook_tools.colour import add_display_normalized_colour_columns
from bragg_onion._notebook_tools.strip_plots import (
    select_best_etaC_by_wavelength,
    plot_best_etaC_colour_strip,
    select_strip_grid,
    plot_colour_strip_grid,
    plot_sweep_heatmap_local,
    prepare_strip_grid,
    summarize_strip_grid_cases,
    plot_colour_strip_grid,
)
from bragg_onion._notebook_tools.rosch_plots import (
    make_rosch_macadam_figures,
    add_highlight_cielab_points,
)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600

print("Notebook helper modules imported successfully.")

In [ ]:
# CELL 1 — PARAMETER SPACE DEFINITION
N_A_VALS = np.linspace(1.01, 1.9, 20)         # material A refractive index
K_VALS   = np.linspace(0.01, 0.1, 10)         # same extinction for A and B
N_B = 1.0                                   # material B refractive index

DESIGN_WAVELENGTHS_NM = np.array([480, 560, 640])
N_LAYERS_VALS = [60]
HALF_ANGLES_DEG = np.array([90])
COLLECTION_NA_VALS = np.sin(np.deg2rad(HALF_ANGLES_DEG))
OUTER_SHELL_THICKNESS_NM_VALS = [
    5,
    10,
    20,
    40,
]
CORE_THICKNESS_FACTORS = [
    0.5,
    1.0,
    2.0,
    4.0,
]

print("n_A_vals:", N_A_VALS)
print("k_vals:", K_VALS)
print("design_wavelengths_nm:", DESIGN_WAVELENGTHS_NM)
print("n_layers_vals:", N_LAYERS_VALS)
print("half_angles_deg:", HALF_ANGLES_DEG)
print("collection_NA_vals:", COLLECTION_NA_VALS)
print("outer_shell_thickness_nm_vals:", OUTER_SHELL_THICKNESS_NM_VALS)
print("core_thickness_factors:", CORE_THICKNESS_FACTORS)

In [ ]:
# CELL 2 — MASTER PARAMETER GRID (PRIMARY SWEEP)

PARAMETER_GRID = {
    "peak_wavelength_m": DESIGN_WAVELENGTHS_NM * 1e-9,
    "n_layers": N_LAYERS_VALS,
    "outer_layer": ["A"],
    "core_thickness_factor": CORE_THICKNESS_FACTORS,
    "outer_shell_thickness_nm": OUTER_SHELL_THICKNESS_NM_VALS
}

PARAMETER_GRID

In [ ]:
# CELL 3 — OUTPUT SOURCE / NOTEBOOK SETTINGS

# Windows path to the copied workstation output directory.
# This can stay in Windows form even when the notebook runs in WSL:
# load_sweep_outputs(...) will convert it automatically if needed.
SOURCE_OUTPUT_DIR_WINDOWS = (
    r"C:\Users\SchwarzN\OneDrive - Université de Fribourg"
    r"\Institution\P1_BraggSpericalPigments\Simulation\MIE_Computation"
    r"\Scattnlay_BraggOnion\outputs\analyse_absorber_location_detailed"
)

# Default colour view for object-colour-style plotting:
DEFAULT_COLOUR_VIEW = "display"   # "raw", "display", or "both"

print("SOURCE_OUTPUT_DIR_WINDOWS:")
print(SOURCE_OUTPUT_DIR_WINDOWS)
print("DEFAULT_COLOUR_VIEW:", DEFAULT_COLOUR_VIEW)

In [ ]:
# CELL 4 — LOAD SWEEP OUTPUTS & ADD DISPLAY-NORMALIZED COLOURS

df_all, OUTPUT_DIR, USE_PARQUET = load_sweep_outputs(
    SOURCE_OUTPUT_DIR_WINDOWS,
    combine_if_needed=True,
    verbose=True,
)

# Working dataframe for plotting/analysis
df_plot = add_display_normalized_colour_columns(df_all, inplace=False)

print("\nLoaded dataframe summary:")
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("USE_PARQUET  :", USE_PARQUET)
print("df_all rows  :", len(df_all))
print("df_plot rows :", len(df_plot))

display(df_plot.head())

In [ ]:
print(df_all.columns)
print(df_plot['core_thickness_factor'].unique())

In [ ]:
print(df_plot.shape)

if "absorber_location" in df_plot.columns:
    print(df_plot["absorber_location"].value_counts())
else:
    print("absorber_location column missing")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameter values
k_values = np.sort(df_all['k'].unique())
nA_values = np.sort(df_all['n_A'].unique())
layer_values = np.sort(df_all['n_layers'].unique())
print(df_all['design_peak_wavelength_nm'].unique())

# Marker for each absorber location
absorber_locations = np.sort(df_all["absorber_location"].unique())

marker_dict = {
    "core": "o",
    "low_index": "s",
    "all_layers": "^",
    "outer_shell": "D",
}

# Small x-offset for each n_A
offsets = np.linspace(-8, 8, len(nA_values))
offset_dict = dict(zip(nA_values, offsets))

# Marker size for each k value
sizes = np.linspace(60, 180, len(k_values))
size_dict = dict(zip(k_values, sizes))

plt.figure(figsize=(10, 6))

for k in k_values:
    # if k != 0.05:
    #     continue

    for nA in nA_values:
        # if nA == 1.5:
        #     continue

        for absorber_location in absorber_locations:
            if absorber_location != "outer_shell": # 'core' 'low_index' 'all_layers' 'outer_shell'
                continue

            subset = df_all[
                (df_all['collection_na'] == 1.0) &
                (df_all['k'] == k) &
                (df_all['n_A'] == nA) &
                (df_all['absorber_location'] == absorber_location)
            ]

            if subset.empty:
                continue

            hue = np.mod(np.degrees(np.arctan2(subset['b'], subset['a'])), 360)
            # Shift hsv colour map
            hue_shift = 290  # degrees

            hue_display = (hue + hue_shift) % 360
            
            plt.scatter(
                subset['design_peak_wavelength_nm'] + offset_dict[nA],
                hue,
                # c=hue_display,
                c=subset['hex'],
                cmap='hsv',
                # marker=marker_dict[subset['absorber_location'].iloc[0]],
                marker=marker_dict[absorber_location],
                s=size_dict[k],
                edgecolor=None,
                label=f'k={k}, nA={nA}, absorber_location={absorber_location}',
                alpha=1
            )

import matplotlib as mpl

ax = plt.gca()      # or better: fig, ax = plt.subplots(...)

norm = mpl.colors.Normalize(vmin=0, vmax=360)
sm = mpl.cm.ScalarMappable(norm=norm, cmap='hsv')
sm.set_array([])
ticks = np.linspace(0, 360, 7)
cbar = plt.colorbar(sm, ax=ax, ticks=ticks)
cbar.set_label("Hue angle [degrees] (hsv colormap)")

plt.xticks(df_all['design_peak_wavelength_nm'].unique())
plt.xlabel('Design Peak Wavelength (nm)')
plt.yticks(np.arange(0, 361, 60))
plt.ylabel('Hue Angle [degrees] (CIE a*b* plane)')
plt.title('Hue Angle vs Design Peak Wavelength')
from matplotlib.lines import Line2D

size_handles = []

for k in k_values:
    size_handles.append(
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor='gray',
            markeredgecolor='k',
            markersize=np.sqrt(size_dict[k]),
            label=f'k = {k:.2f}'
        )
    )

shape_handles = []

for loc in absorber_locations:
    shape_handles.append(
        Line2D(
            [0], [0],
            marker=marker_dict[loc],
            linestyle='None',
            color='k',
            markersize=8,
            label=loc.replace('_', ' ')
        )
    )

leg1 = plt.legend(
    handles=size_handles,
    title='Extinction k',
    loc='upper right'
)

plt.gca().add_artist(leg1)

plt.legend(
    handles=shape_handles,
    title='Absorber location',
    loc='lower right'
)

plt.text(
    0.02, 0.2,
    "Horizontal offset indicates $n_A$\n"
    "Marker size indicates $k$\n"
    "Marker shape indicates absorber location",
    transform=plt.gca().transAxes,
    va='top',
    fontsize=10,
    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray')
)

# possible target hues
target_hues = np.array([30, 60, 90, 150, 210, 270, 315])
# corresponding target wavelengths
target_wavelengths = np.array([760, 680, 620, 540, 500, 440, 380])
for target_hue in target_hues:
    plt.axhline(
        y=target_hue,
        color='gray',
        linestyle='--',
        linewidth=0.8,
        alpha=0.5
    )

plt.show()

In [ ]:
# CELL 5 — MAIN PLOT
# PART A — ROSCH–MACADAM COLOUR SOLID
# ============================================================

ROSCH_COLOUR_VIEW = "both"   # "raw", "display", or "both"

generated_figures = make_rosch_macadam_figures(
    df_plot,
    colour_view=ROSCH_COLOUR_VIEW,
    max_overlay_points=2000,
    fixed_marker_size=4.0,
    marker_symbol_variable="n_layers",   # change freely
    marker_symbol_style="solid_and_line",  # "solid_only" or "solid_and_line"
    marker_edge_color="black",           # or None
    marker_edge_width=0.4,
    optimal_marker_size=4.8,
    optimal_marker_edge_color=None,
    optimal_marker_edge_width=0.7,
    save_html=True,
    output_dir=OUTPUT_DIR,
    show_figures=True,
)

print("Generated Rosch–MacAdam figure keys:", list(generated_figures.keys()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

group_cols = [
    "absorber_location",
    "design_peak_wavelength_nm",
]

idx = (
    df_all
    .groupby(group_cols)["eta_C"]
    .idxmax()
)

df_best = df_all.loc[idx].copy()

locations = [
    "core",
    "low_index",
    "all_layers",
    "outer_shell",
]

wavelengths = sorted(df_best["design_peak_wavelength_nm"].unique())

fig, ax = plt.subplots(
    figsize=(3.0 * len(wavelengths),
             2.3 * len(locations))
)

for i, loc in enumerate(locations):
    for j, wl in enumerate(wavelengths):

        row = df_best[
            (df_best["absorber_location"] == loc)
            & (df_best["design_peak_wavelength_nm"] == wl)
        ]

        if row.empty:
            continue

        row = row.iloc[0]

        rect = Rectangle(
            (j, len(locations)-1-i),
            1,
            1,
            facecolor=row["hex"],
            edgecolor="black",
        )

        ax.add_patch(rect)

        if loc == "core":
            txt = (
                f"ηC={100*row['eta_C']:.0f}%\n"
                f"ηL={100*row['eta_L']:.0f}%\n"
                f"nA={row['n_A']:.1f}\n"
                f"k={row['k']:.2f}\n"
                f"d core factor={row['core_thickness_factor']:.1f}\n"
            )
        elif loc == "outer_shell":
            txt = (
                f"ηC={100*row['eta_C']:.0f}%\n"
                f"ηL={100*row['eta_L']:.0f}%\n"
                f"nA={row['n_A']:.1f}\n"
                f"k={row['k']:.2f}\n"
                f"d shell={row['outer_shell_thickness_nm']:.0f} nm"
            )
        else:
            txt = (
                f"ηC={100*row['eta_C']:.0f}%\n"
                f"ηL={100*row['eta_L']:.0f}%\n"
                f"nA={row['n_A']:.1f}\n"
                f"k={row['k']:.2f}"
            )

        ax.text(
            j + 0.5,
            len(locations)-1-i + 0.5,
            txt,
            ha="center",
            va="center",
            fontsize=9,
            color="black",
        )

ax.set_xlim(0, len(wavelengths))
ax.set_ylim(0, len(locations))

ax.set_xticks(np.arange(len(wavelengths)) + 0.5)
ax.set_xticklabels([f"{w:.0f} nm" for w in wavelengths])

ax.set_yticks(np.arange(len(locations)) + 0.5)
ax.set_yticklabels(locations[::-1])

ax.set_title("Best ηC case for each absorber location and target wavelength")

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

heatmap_data = (
    df_best
    .pivot(
        index="absorber_location",
        columns="design_peak_wavelength_nm",
        values="eta_C",
    )
)

plt.figure(figsize=(7,4))

sns.heatmap(
    100 * heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="viridis",
    cbar_kws={"label": "ηC [%]"},
)

plt.xlabel("Design wavelength (nm)")
plt.ylabel("Absorber location")
plt.title("Maximum chroma achieved by each absorber location")

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

marker_dict = {
    "core": "o",
    "low_index": "s",
    "all_layers": "^",
    "outer_shell": "D",
}

size_dict = {
    k: s
    for k, s in zip(
        sorted(df_all["k"].unique()),
        np.linspace(50, 250, df_all["k"].nunique())
    )
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15,5),
    sharex=True,
    sharey=True,
)

for ax, wl in zip(
    axes,
    sorted(df_all["design_peak_wavelength_nm"].unique())
):

    subset_wl = df_all[
        df_all["design_peak_wavelength_nm"] == wl
    ]

    for loc in subset_wl["absorber_location"].unique():

        sub = subset_wl[
            subset_wl["absorber_location"] == loc
        ]

        ax.scatter(
            100 * sub["eta_L"],
            100 * sub["eta_C"],
            c=sub["hex"],
            s=[size_dict[v] for v in sub["k"]],
            marker=marker_dict[loc],
            edgecolors="k",
            alpha=0.8,
            label=loc,
        )

    ax.set_title(f"{wl:.0f} nm")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("ηC [%]")

for ax in axes:
    ax.set_xlabel("ηL [%]")

handles = []

for loc, marker in marker_dict.items():
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker=marker,
            color="k",
            linestyle="None",
            markersize=8,
            label=loc,
        )
    )

fig.legend(
    handles=handles,
    loc="lower center",
    ncols=2,
    bbox_to_anchor=(0.5, 0.7)
)

fig.suptitle(
    "Chroma–lightness tradeoff for different absorber placements",
    y=1.05
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PART B — COLOUR STRIP OPTION 1
# Best η_C case for each design wavelength
# ============================================================

fixed_filters_best_strip = {
    # "collection_angle_deg": 90.0,
    # "n_layers": 100,
    # "n_A": 2.1,
    "k": 0.05,
}

df_best_strip = select_best_etaC_by_wavelength(
    df_plot,   # use df_plot if you want hex_display
    fixed_filters=fixed_filters_best_strip,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
)

plot_best_etaC_colour_strip(
    df_best_strip,
    x="design_peak_wavelength_nm",
    colour_hex_col="hex",   # or "hex"
    annotate_cols=["eta_C", "eta_L"],
    title="Best η_C case for each design wavelength\n(fixed k = 0.05)",
)

plt.show()

# Print properties of the best strip dataframe for reference:
df_best_strip_print = df_best_strip.copy().sort_values("design_peak_wavelength_nm")

if "eta_C" in df_best_strip_print.columns:
    df_best_strip_print["eta_C_pct"] = (100 * df_best_strip_print["eta_C"]).round(0).astype(int)

if "eta_L" in df_best_strip_print.columns:
    df_best_strip_print["eta_L_pct"] = (100 * df_best_strip_print["eta_L"]).round(0).astype(int)

cols_to_show = [
    "design_peak_wavelength_nm",
    "case_index",
    "eta_C_pct",
    "eta_L_pct",
    "n_layers",
    "n_A",
    "k",
    "collection_angle_deg",
    "outer_layer",
    "hex_display",
]

cols_to_show = [c for c in cols_to_show if c in df_best_strip_print.columns]

display(df_best_strip_print[cols_to_show])

In [ ]:
# ============================================================
# PART C — COLOUR STRIP OPTION 2
# Multi-row strip grid:
# rows = full range of one sweep parameter
# columns = design wavelength
# ============================================================


# ------------------------------------------------------------
# User choices
# ------------------------------------------------------------
row_variable = "n_A"   # e.g. "k", "n_layers", "collection_angle_deg", "n_A"
colour_hex_col = "hex"                  # or "hex_display"

fixed_filters_grid = {
    "collection_angle_deg": 90,   # removed automatically if row_variable matches
    # "n_layers": 100,
    "n_A": 2.1,
    "k": 0.05,
}

# ------------------------------------------------------------
# Prepare data + auto-generated metadata
# ------------------------------------------------------------
df_grid, grid_meta = prepare_strip_grid(
    df_all,
    df_plot,
    row_variable=row_variable,
    fixed_filters=fixed_filters_grid,
    colour_hex_col=colour_hex_col,
    x="design_peak_wavelength_nm",
    score_col="eta_C",
    select_best_within_cell=True,
)

# Optional info messages
if grid_meta["removed_filter"] is not None:
    print(
        f"Removed '{row_variable}={grid_meta['removed_filter']}' from fixed_filters "
        f"because the row variable must vary across rows."
    )

print("Fixed filters used:")
print(grid_meta["fixed_filters_clean"])
print(f"Row variable : {grid_meta['row_variable']}")
print(f"Colour column: {grid_meta['colour_hex_col']}")
print(f"Source df    : {grid_meta['df_source_name']}")
print(f"Selected tile cases: {len(df_grid)}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plot_colour_strip_grid(
    df_grid,
    row_variable=grid_meta["row_variable"],
    x="design_peak_wavelength_nm",
    colour_hex_col=grid_meta["colour_hex_col"],
    annotate_cols=["eta_C", "eta_L"],
    row_label_fmt=grid_meta["row_label_fmt"],
    title=grid_meta["title"],
    overlay_line_cols=["eta_C", "eta_L"],
    overlay_line_colors={"eta_C": "black", "eta_L": "white"},
    overlay_line_scale="global_shared",
    overlay_reference_values=[0.0, 1.0],
)

plt.show()

# ------------------------------------------------------------
# Compact summary table of the plotted tiles
# ------------------------------------------------------------
df_grid_print = summarize_strip_grid_cases(
    df_grid,
    row_variable=grid_meta["row_variable"],
    fixed_filters=grid_meta["fixed_filters_clean"],
    colour_hex_col=grid_meta["colour_hex_col"],
    x="design_peak_wavelength_nm",
)

display(df_grid_print.sort_values("design_peak_wavelength_nm"))

In [ ]:
from matplotlib.patches import Circle

# (1.2) CIELAB a*–b* plane
plt.figure(figsize=(6,6))
plt.scatter(df_all["a"], df_all["b"], c=df_all["hex"], s=15)
plt.axhline(0, color="gray")
plt.axvline(0, color="gray")
plt.xlabel("a*")
plt.ylabel("b*")
plt.title("CIELAB a*–b* plane")
plt.axis("equal")
# add radial rings to indicate hue angles and chroma levels
hue_angles = np.arange(0, 360, 30)
chroma_levels = np.arange(0.0, 75.0, 20.0)
for hue in hue_angles:
    plt.plot(
        chroma_levels * np.cos(np.radians(hue)),
        chroma_levels * np.sin(np.radians(hue)),
        color="k",
        linestyle="--",
        linewidth=0.5,
        alpha=0.5,
    )
# add rings for chroma levels
for chroma in chroma_levels:
    circle = Circle((0, 0), float(chroma), color="gray", fill=False, linestyle="--", linewidth=0.5, alpha=0.5)
    plt.gca().add_artist(circle)
plt.show()

Analysis extended to forward scattering

In [ ]:
from bragg_onion.geometry import (
    ExtinctionModifier,
    ExtraOuterShellSpec,
)
from bragg_onion.materials import ConstantDispersion


def build_absorber_objects(row):
    """
    Reconstruct extinction_modifiers / extra_outer_shell
    from one dataframe row.
    """

    k = float(row["k"])
    absorber_location = row["absorber_location"]

    extinction_modifiers = None
    extra_outer_shell = None

    if absorber_location == "core":

        extinction_modifiers = [
            ExtinctionModifier(
                target="core",
                mode="substitute",
                profile=k,
                name=f"core_k_{k:.3f}",
            )
        ]

    elif absorber_location == "low_index":

        extinction_modifiers = [
            ExtinctionModifier(
                target="material_B",
                mode="substitute",
                profile=k,
                name=f"low_index_k_{k:.3f}",
            )
        ]

    elif absorber_location == "all_layers":

        extinction_modifiers = [
            ExtinctionModifier(
                target="both_materials",
                mode="substitute",
                profile=k,
                name=f"all_layers_k_{k:.3f}",
            )
        ]

    elif absorber_location == "outer_shell":

        absorber_material = ConstantDispersion.from_nk(
            "outer_absorber",
            n=1.0,
            k=k,
        )

        extra_outer_shell = ExtraOuterShellSpec(
            thickness_m=float(row["outer_shell_thickness_nm"]) * 1e-9,
            material=absorber_material,
            name=f"outer_shell_k_{k:.3f}",
        )

    else:
        raise ValueError(
            f"Unknown absorber location: {absorber_location}"
        )

    return extinction_modifiers, extra_outer_shell

In [ ]:
import numpy as np

from bragg_onion.materials import ConstantDispersion
from bragg_onion.sweep import run_bragg_onion_sweep
from bragg_onion._notebook_tools.scattering_extract import (
    extract_case_arrays,
)

analysis_wavelengths_nm = np.arange(380.0, 781.0, 1.0)
analysis_theta_deg = np.linspace(0.0, 180.0, 721)

best_cases = {}

for _, row_case in df_best.iterrows():

    # --------------------------------------------------------
    # Reconstruct the base, nonabsorbing Bragg materials
    # Absorption is added separately according to location.
    # --------------------------------------------------------
    material_a = ConstantDispersion.from_nk(
        "A",
        n=float(row_case["n_A"]),
        k=0.0,
    )

    material_b = ConstantDispersion.from_nk(
        "B",
        n=float(row_case["n_B"]),
        k=0.0,
    )

    # --------------------------------------------------------
    # Reconstruct absorber placement
    # --------------------------------------------------------
    extinction_modifiers, extra_outer_shell = (
        build_absorber_objects(row_case)
    )

    # --------------------------------------------------------
    # Reconstruct exactly one geometry case
    # --------------------------------------------------------
    parameter_grid = {
        "peak_wavelength_m": [
            float(row_case["peak_wavelength_m"])
        ],
        "n_layers": [
            int(row_case["n_layers"])
        ],
        "outer_layer": [
            str(row_case["outer_layer"])
        ],
        "core_thickness_factor": [
            float(row_case["core_thickness_factor"])
        ],
    }

    # --------------------------------------------------------
    # Detailed rerun
    # The backward result reproduces the original sweep setup.
    # --------------------------------------------------------
    sweep_case = run_bragg_onion_sweep(
        geometry_mode="peak_wavelength",
        parameter_grid=parameter_grid,
        material_a=material_a,
        material_b=material_b,
        n_medium=1.0,
        wavelengths_m=analysis_wavelengths_nm * 1e-9,
        theta_rad=np.deg2rad(analysis_theta_deg),
        collection_na=float(row_case["collection_na"]),
        collection_direction="backward",
        integration_quantity_for_colour="q_collected_geom",
        colour_wavelength_min_nm=None,
        colour_wavelength_max_nm=None,
        colour_normalize_input=False,
        colour_normalization="max",
        extinction_modifiers=extinction_modifiers,
        extra_outer_shell=extra_outer_shell,
        store_full_results=True,
        progress=False,
    )

    data = extract_case_arrays(sweep_case)

    key = (
        str(row_case["absorber_location"]),
        int(round(float(row_case["design_peak_wavelength_nm"]))),
    )

    if key in best_cases:
        raise ValueError(
            f"Duplicate best-case key encountered: {key}"
        )

    best_cases[key] = {
        "row": row_case.copy(),
        "sweep_case": sweep_case,
        "case_result": sweep_case.case_results[0],
        "data": data,
    }

    print(
        f"Finished {key}: "
        f"n_A={float(row_case['n_A']):.3f}, "
        f"k={float(row_case['k']):.3f}"
    )

print(f"\nStored {len(best_cases)} detailed rerun cases.")
print("Keys:")
print(list(best_cases.keys()))

In [ ]:
# RERUN THE SAME 12 BEST CASES WITH FORWARD COLLECTION

import numpy as np

from bragg_onion.materials import ConstantDispersion
from bragg_onion.sweep import run_bragg_onion_sweep
from bragg_onion._notebook_tools.scattering_extract import (
    extract_case_arrays,
)

best_cases_forward = {}

for _, row_case in df_best.iterrows():

    absorber_location = str(row_case["absorber_location"])
    design_wavelength_nm = int(
        round(float(row_case["design_peak_wavelength_nm"]))
    )

    key = (
        absorber_location,
        design_wavelength_nm,
    )

    print(
        f"Running forward collection: "
        f"location={absorber_location}, "
        f"design λ={design_wavelength_nm} nm"
    )

    # --------------------------------------------------------
    # Reconstruct the nonabsorbing base materials.
    # Absorption is introduced by build_absorber_objects().
    # --------------------------------------------------------
    material_a = ConstantDispersion.from_nk(
        "A",
        n=float(row_case["n_A"]),
        k=0.0,
    )

    material_b = ConstantDispersion.from_nk(
        "B",
        n=float(row_case["n_B"]),
        k=0.0,
    )

    # --------------------------------------------------------
    # Reconstruct absorber placement and strength.
    # --------------------------------------------------------
    extinction_modifiers, extra_outer_shell = (
        build_absorber_objects(row_case)
    )

    # --------------------------------------------------------
    # Reconstruct exactly one geometry.
    # Explicit casts are important for values read from pandas.
    # --------------------------------------------------------
    parameter_grid = {
        "peak_wavelength_m": [
            float(row_case["peak_wavelength_m"])
        ],
        "n_layers": [
            int(row_case["n_layers"])
        ],
        "outer_layer": [
            str(row_case["outer_layer"])
        ],
        "core_thickness_factor": [
            float(row_case["core_thickness_factor"])
        ],
    }

    # --------------------------------------------------------
    # Forward-collected rerun.
    # --------------------------------------------------------
    sweep_case_forward = run_bragg_onion_sweep(
        geometry_mode="peak_wavelength",
        parameter_grid=parameter_grid,
        material_a=material_a,
        material_b=material_b,
        n_medium=1.0,
        wavelengths_m=np.asarray(
            analysis_wavelengths_nm,
            dtype=float,
        ) * 1e-9,
        theta_rad=np.deg2rad(
            np.asarray(analysis_theta_deg, dtype=float)
        ),
        collection_na=float(row_case["collection_na"]),
        collection_direction="forward",
        integration_quantity_for_colour="q_collected_geom",
        colour_wavelength_min_nm=None,
        colour_wavelength_max_nm=None,
        colour_normalize_input=False,
        colour_normalization="max",
        extinction_modifiers=extinction_modifiers,
        extra_outer_shell=extra_outer_shell,
        store_full_results=True,
        progress=False,
    )

    data_forward = extract_case_arrays(sweep_case_forward)

    best_cases_forward[key] = {
        "row": row_case.copy(),
        "sweep_case": sweep_case_forward,
        "case_result": sweep_case_forward.case_results[0],
        "data": data_forward,
    }

    print(f"  Finished {key}")

print(
    f"\nStored {len(best_cases_forward)} "
    "forward-collected detailed cases."
)

print("Forward-case keys:")
print(list(best_cases_forward.keys()))

In [ ]:
# COMMON SETUP FOR FORWARD/BACKWARD COMPARISON PLOTS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from bragg_onion.sweep import make_plotting_aliases


location_order = [
    "core",
    "low_index",
    "all_layers",
    "outer_shell",
]

location_labels = {
    "core": "Core",
    "low_index": "Low-index layers",
    "all_layers": "All Bragg layers",
    "outer_shell": "Outer shell",
}

location_styles = {
    "core": "-",
    "low_index": "--",
    "all_layers": "-.",
    "outer_shell": ":",
}

design_wavelengths = sorted(
    {
        int(round(float(key[1])))
        for key in best_cases.keys()
    }
)

# Confirm corresponding forward and backward results exist.
backward_keys = set(best_cases.keys())
forward_keys = set(best_cases_forward.keys())

if backward_keys != forward_keys:
    raise ValueError(
        "Forward and backward case dictionaries do not contain identical cases.\n"
        f"Missing from forward: {sorted(backward_keys - forward_keys)}\n"
        f"Missing from backward: {sorted(forward_keys - backward_keys)}"
    )

print("Absorber locations:", location_order)
print("Design wavelengths:", design_wavelengths)
print("Number of corresponding cases:", len(backward_keys))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from matplotlib.patches import Rectangle
from bragg_onion.sweep import make_plotting_aliases


def readable_text_colour(hex_colour):
    """Choose black or white annotation text from tile luminance."""
    rgb = np.asarray(
        mpl.colors.to_rgb(hex_colour),
        dtype=float,
    )

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.52 else "white"


# ------------------------------------------------------------
# Extract and validate the forward-collected colour results
# ------------------------------------------------------------
forward_summary_rows = {}

for key, result in best_cases_forward.items():

    sweep_case_forward = result["sweep_case"]
    case_result_forward = result["case_result"]
    data_forward = result["data"]

    # IMPORTANT:
    # make_plotting_aliases creates the convenient columns
    # "hex", "eta_C", "eta_L", etc.
    forward_df = make_plotting_aliases(
        sweep_case_forward.dataframe
    )

    if len(forward_df) != 1:
        raise ValueError(
            f"Expected exactly one forward dataframe row for {key}, "
            f"but found {len(forward_df)}."
        )

    if "hex" not in forward_df.columns:
        raise KeyError(
            f"The forward dataframe for {key} still has no 'hex' alias.\n"
            f"Available columns:\n{forward_df.columns.tolist()}"
        )

    forward_row = forward_df.iloc[0]

    # Validate that the forward colour spectrum is the same quantity
    # as the forward-collected q_collected_geom spectrum.
    colour_result = case_result_forward.colour_result

    spectrum_used = np.asarray(
        colour_result.spectrum_used,
        dtype=float,
    ).ravel()

    q_forward = np.asarray(
        data_forward["q_collected_geom"],
        dtype=float,
    ).ravel()

    wavelengths_colour_nm = np.asarray(
        colour_result.wavelengths_nm,
        dtype=float,
    ).ravel()

    wavelengths_integrated_nm = np.asarray(
        data_forward["wavelengths_int_nm"],
        dtype=float,
    ).ravel()

    if not np.allclose(
        wavelengths_colour_nm,
        wavelengths_integrated_nm,
    ):
        raise ValueError(
            f"Colour and integrated wavelength grids differ for {key}."
        )

    if not np.allclose(
        spectrum_used,
        q_forward,
        equal_nan=True,
    ):
        raise ValueError(
            f"The forward spectrum_used does not match "
            f"q_collected_geom for {key}."
        )

    # This is the hex colour calculated from the forward spectrum.
    tile_hex = str(forward_row["hex_display"])

    if not mpl.colors.is_color_like(tile_hex):
        raise ValueError(
            f"Invalid forward hex colour for {key}: {tile_hex!r}"
        )

    forward_summary_rows[key] = {
        "summary_row": forward_row,
        "tile_hex": tile_hex,
        "spectrum_used": spectrum_used,
        "wavelengths_nm": wavelengths_colour_nm,
    }


print(
    f"Validated forward spectra and colours for "
    f"{len(forward_summary_rows)} cases."
)

In [ ]:
# PANEL 1 — FORWARD-COLLECTED COLOUR-STRIP MATRIX
#
# Each tile corresponds to the same parameter combination that maximized
# backward eta_C for that absorber location and design wavelength.
# The displayed colour and metrics are recalculated for forward collection.

def readable_text_colour(hex_colour):
    """
    Choose black or white text based on displayed tile luminance.
    """
    try:
        rgb = np.asarray(
            mpl.colors.to_rgb(hex_colour),
            dtype=float,
        )
        luminance = (
            0.2126 * rgb[0]
            + 0.7152 * rgb[1]
            + 0.0722 * rgb[2]
        )
        return "black" if luminance > 0.52 else "white"
    except Exception:
        return "black"


# Extract one forward summary row for every corresponding case.
forward_summary_rows = {}

for key, result in best_cases_forward.items():

    # Add short plotting aliases such as eta_C, eta_L and hex.
    forward_df = make_plotting_aliases(
        result["sweep_case"].dataframe
    )

    # Add display-normalized CIELAB, XYZ and hex columns.
    forward_df = add_display_normalized_colour_columns(
        forward_df,
        inplace=False,
    )

    if len(forward_df) != 1:
        raise ValueError(
            f"Expected one forward dataframe row for {key}, "
            f"but found {len(forward_df)}."
        )

    required_columns = [
        "eta_C",
        "eta_L",
        "hex",
        "hex_display",
        "L_display",
        "a_display",
        "b_display",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in forward_df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Forward dataframe for {key} is missing columns: "
            f"{missing_columns}"
        )

    forward_summary_rows[key] = forward_df.iloc[0]


fig, ax = plt.subplots(
    figsize=(
        3.2 * len(design_wavelengths),
        2.5 * len(location_order),
    )
)

n_locations = len(location_order)

for i, location in enumerate(location_order):
    y = n_locations - 1 - i

    for j, design_wl in enumerate(design_wavelengths):
        key = (location, design_wl)

        if key not in best_cases_forward:
            continue

        original_row = best_cases_forward[key]["row"]
        forward_row = forward_summary_rows[key]

        tile_hex = str(forward_row["hex_display"])

        rect = Rectangle(
            (j, y),
            1,
            1,
            facecolor=tile_hex,
            edgecolor="black",
            linewidth=1.0,
        )
        ax.add_patch(rect)

        text_lines = [
            f"ηC,fwd={100 * float(forward_row['eta_C']):.0f}%",
            f"ηL,fwd={100 * float(forward_row['eta_L']):.0f}%",
            f"nA={float(original_row['n_A']):.2f}",
            f"k={float(original_row['k']):.3f}",
        ]

        if location == "core":
            text_lines.append(
                "core factor="
                f"{float(original_row['core_thickness_factor']):.2f}"
            )

        elif location == "outer_shell":
            if "outer_shell_thickness_nm" in original_row.index:
                text_lines.append(
                    "shell="
                    f"{float(original_row['outer_shell_thickness_nm']):.0f} nm"
                )

        ax.text(
            j + 0.5,
            y + 0.5,
            "\n".join(text_lines),
            ha="center",
            va="center",
            fontsize=9,
            color=readable_text_colour(tile_hex),
        )


ax.set_xlim(0, len(design_wavelengths))
ax.set_ylim(0, len(location_order))
ax.set_aspect("equal")

ax.set_xticks(
    np.arange(len(design_wavelengths)) + 0.5
)
ax.set_xticklabels(
    [f"{wl:.0f} nm" for wl in design_wavelengths]
)

ax.set_yticks(
    np.arange(len(location_order)) + 0.5
)
ax.set_yticklabels(
    [
        location_labels[loc]
        for loc in location_order[::-1]
    ]
)

ax.set_xlabel("Design wavelength")
ax.set_ylabel("Absorber location")
ax.set_title(
    "Forward-collected colours for the cases with best backward ηC\n"
    "(NA = 1; identical physical cases used for forward and backward comparison)"
)

fig.tight_layout()
plt.show()

In [ ]:
# PANEL 2 — FORWARD AND BACKWARD COLLECTED SPECTRA

NORMALIZE_SPECTRA = False
# False: compare absolute q_collected_geom values.
# True: normalize each individual spectrum to its own maximum,
#       emphasizing shape rather than magnitude.

forward_colour = "tab:blue"
backward_colour = "tab:red"


def normalize_spectrum_if_requested(values, normalize):
    values = np.asarray(values, dtype=float).ravel()

    if not normalize:
        return values

    maximum = np.nanmax(values)
    return values / maximum if maximum > 0 else values


fig, axes = plt.subplots(
    1,
    len(design_wavelengths),
    figsize=(5.2 * len(design_wavelengths), 4.8),
    sharex=True,
    sharey=NORMALIZE_SPECTRA,
)

axes = np.atleast_1d(axes)

for ax, design_wl in zip(axes, design_wavelengths):

    for location in location_order:
        key = (location, design_wl)

        backward_data = best_cases[key]["data"]
        forward_data = best_cases_forward[key]["data"]

        wl_backward = np.asarray(
            backward_data["wavelengths_int_nm"],
            dtype=float,
        ).ravel()

        wl_forward = np.asarray(
            forward_data["wavelengths_int_nm"],
            dtype=float,
        ).ravel()

        q_backward = normalize_spectrum_if_requested(
            backward_data["q_collected_geom"],
            NORMALIZE_SPECTRA,
        )

        q_forward = normalize_spectrum_if_requested(
            forward_data["q_collected_geom"],
            NORMALIZE_SPECTRA,
        )

        linestyle = location_styles[location]

        ax.plot(
            wl_backward,
            q_backward,
            color=backward_colour,
            linestyle=linestyle,
            linewidth=2.0,
        )

        ax.plot(
            wl_forward,
            q_forward,
            color=forward_colour,
            linestyle=linestyle,
            linewidth=2.0,
        )

    ax.axvline(
        design_wl,
        color="gray",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
    )

    ax.set_title(f"Design λ = {design_wl:.0f} nm")
    ax.set_xlabel("Wavelength (nm)")
    ax.set_xlim(
        min(wl_backward.min(), wl_forward.min()),
        max(wl_backward.max(), wl_forward.max()),
    )
    ax.grid(True, alpha=0.3)

    if NORMALIZE_SPECTRA:
        ax.set_ylim(0, 1.05)

if NORMALIZE_SPECTRA:
    axes[0].set_ylabel(
        "Collected spectrum\n(normalized independently)"
    )
else:
    axes[0].set_ylabel(
        r"Collected scattering efficiency $Q_{\mathrm{collected,geom}}$"
    )


# Legend part 1: collection direction
direction_handles = [
    Line2D(
        [0], [0],
        color=forward_colour,
        linewidth=2.5,
        label="Forward collection",
    ),
    Line2D(
        [0], [0],
        color=backward_colour,
        linewidth=2.5,
        label="Backward collection",
    ),
]

# Legend part 2: absorber location
location_handles = [
    Line2D(
        [0], [0],
        color="black",
        linestyle=location_styles[location],
        linewidth=2.0,
        label=location_labels[location],
    )
    for location in location_order
]

fig.legend(
    handles=direction_handles + location_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
    ncol=3,
    frameon=True,
)

fig.suptitle(
    "Forward- and backward-collected spectra of the same best-ηC cases",
    y=1.16,
)

fig.tight_layout()
plt.show()

In [ ]:
# PANEL 3 — ASYMMETRY g AND ALBEDO

location_colours = {
    "core": "tab:blue",
    "low_index": "tab:orange",
    "all_layers": "tab:green",
    "outer_shell": "tab:red",
}

# ------------------------------------------------------------
# Verify that g and albedo are independent of collection direction.
# ------------------------------------------------------------
for key in best_cases:
    backward_data = best_cases[key]["data"]
    forward_data = best_cases_forward[key]["data"]

    if not np.allclose(
        backward_data["g"],
        forward_data["g"],
        equal_nan=True,
    ):
        raise ValueError(
            f"Forward/backward g mismatch for {key}."
        )

    if not np.allclose(
        backward_data["albedo"],
        forward_data["albedo"],
        equal_nan=True,
    ):
        raise ValueError(
            f"Forward/backward albedo mismatch for {key}."
        )

print("Verified: g and albedo agree between forward and backward reruns.")


# ------------------------------------------------------------
# Plot.
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    len(design_wavelengths),
    figsize=(5.4 * len(design_wavelengths), 4.8),
    sharex=True,
    sharey=True,
)

axes = np.atleast_1d(axes)
right_axes = []

for ax, design_wl in zip(axes, design_wavelengths):
    ax_right = ax.twinx()
    right_axes.append(ax_right)

    for location in location_order:
        key = (location, design_wl)
        data_case = best_cases[key]["data"]

        wavelength_nm = np.asarray(
            data_case["wavelengths_nm"],
            dtype=float,
        ).ravel()

        g_values = np.asarray(
            data_case["g"],
            dtype=float,
        ).ravel()

        albedo_values = np.asarray(
            data_case["albedo"],
            dtype=float,
        ).ravel()

        colour = location_colours[location]

        # g on left axis
        ax.plot(
            wavelength_nm,
            g_values,
            color=colour,
            linestyle="-",
            linewidth=2.0,
        )

        # albedo on right axis
        ax_right.plot(
            wavelength_nm,
            albedo_values,
            color=colour,
            linestyle="--",
            linewidth=1.8,
            alpha=0.9,
        )

    ax.axvline(
        design_wl,
        color="gray",
        linestyle=":",
        linewidth=1.0,
    )

    ax.set_title(f"Design λ = {design_wl:.0f} nm")
    ax.set_xlabel("Wavelength (nm)")
    ax.set_xlim(wavelength_nm[0], wavelength_nm[-1])
    ax.grid(True, alpha=0.3)

    # Physical axis ranges
    ax.set_ylim(-1.0, 1.0)
    ax_right.set_ylim(0.0, 1.05)

# Label only outer axes to reduce clutter
axes[0].set_ylabel("Asymmetry parameter g")
right_axes[-1].set_ylabel("Single-scattering albedo")

# Location legend: colour encoding
location_handles = [
    Line2D(
        [0], [0],
        color=location_colours[location],
        linewidth=2.5,
        label=location_labels[location],
    )
    for location in location_order
]

# Quantity legend: line-style encoding
quantity_handles = [
    Line2D(
        [0], [0],
        color="black",
        linestyle="-",
        linewidth=2.0,
        label="Asymmetry g",
    ),
    Line2D(
        [0], [0],
        color="black",
        linestyle="--",
        linewidth=2.0,
        label="Albedo",
    ),
]

fig.legend(
    handles=location_handles + quantity_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.08),
    ncol=3,
    frameon=True,
)

fig.suptitle(
    "Asymmetry and albedo of the best backward-ηC cases",
    y=1.16,
)

fig.tight_layout()
plt.show()

In [ ]:
# PARAMETER SUMMARY FOR THE 12 COMPARED CASES

parameter_columns = [
    "absorber_location",
    "design_peak_wavelength_nm",
    "n_A",
    "n_B",
    "k",
    "n_layers",
    "core_thickness_factor",
    "outer_shell_thickness_nm",
    "eta_C",
    "eta_L",
]

parameter_columns = [
    column
    for column in parameter_columns
    if column in df_best.columns
]

df_case_summary = (
    df_best[parameter_columns]
    .sort_values(
        [
            "design_peak_wavelength_nm",
            "absorber_location",
        ]
    )
    .reset_index(drop=True)
)

display(df_case_summary)
